python 3.11, miniconda, cuda 12.8
ffmpeg==8.0.1
torch==2.9.0
torchcodec==0.9.1

```bash
conda install conda-forge::ffmpeg==8.0.1
ffmpeg -decoders | grep -i nvidia
```

```bash
pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cu128
pip install torchcodec==0.9.1 --index-url=https://download.pytorch.org/whl/cu128
```


In [1]:
import os
import re
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

### dataset 

In [2]:
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "./assets/asr_2026")).resolve()
DATA_ROOT

PosixPath('/mnt/e/work/itmo/subjects/speech_course/itmo-sc-tasks/assignments/g1/assets/asr_2026')

#### parse csv

In [3]:
def read_meta(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # filename,transcription,spk_id,gender,ext,samplerate
    df["filename"] = df["filename"].astype(str)
    df["transcription"] = df["transcription"].astype(str)
    return df

In [4]:
train_csv = DATA_ROOT / "train.csv"
dev_csv   = DATA_ROOT / "dev.csv"

In [5]:
train_df = read_meta(train_csv)
dev_df   = read_meta(dev_csv)

In [6]:
display(train_df.head(3))
display(dev_df.head(3))

print("train:", len(train_df), "dev:", len(dev_df))
print("train speakers:", sorted(train_df.spk_id.unique()))
print("dev speakers:", sorted(dev_df.spk_id.unique()))

,filename,transcription,spk_id,gender,ext,samplerate
0,train/0007c21c23.wav,139473,spk_E,female,wav,24000
1,train/000bee1b1d.wav,992597,spk_B,male,wav,24000
2,train/001a718902.wav,500869,spk_A,female,wav,22050


,filename,transcription,spk_id,gender,ext,samplerate
0,dev/0025d6d9a9.wav,849905,spk_J,male,wav,16000
1,dev/0030479d60.mp3,967653,spk_I,female,mp3,16000
2,dev/003085d7a4.mp3,427524,spk_K,female,mp3,16000


train: 12553 dev: 2265
train speakers: ['spk_A', 'spk_B', 'spk_C', 'spk_D', 'spk_E', 'spk_F']
dev speakers: ['spk_A', 'spk_B', 'spk_C', 'spk_D', 'spk_E', 'spk_F', 'spk_H', 'spk_I', 'spk_J', 'spk_K']


In [7]:
# inD / ooD speaker split
TRAIN_SPK = set(train_df.spk_id.unique().tolist())

def domain_of_spk(spk_id: str) -> str:
    return "inD" if spk_id in TRAIN_SPK else "ooD"

dev_df = dev_df.copy()
dev_df["domain"] = dev_df["spk_id"].map(domain_of_spk)

print(dev_df["domain"].value_counts())
display(dev_df.groupby(["domain","spk_id"]).size().reset_index(name="n").sort_values(["domain","n"], ascending=[True, False]))

domain
ooD    1665
inD     600
Name: count, dtype: int64


,domain,spk_id,n
0,inD,spk_A,100
1,inD,spk_B,100
2,inD,spk_C,100
3,inD,spk_D,100
4,inD,spk_E,100
5,inD,spk_F,100
6,ooD,spk_H,633
7,ooD,spk_I,497
8,ooD,spk_J,344
9,ooD,spk_K,191


In [9]:
# text normalization
_RU_ALLOWED = re.compile(r"[^а-яё0-9\s-]+")

def normalize_ru_text(s: str) -> str:
    s = s.lower().strip()
    s = s.replace("ё", "е")
    s = s.replace("-", " ")
    s = _RU_ALLOWED.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [10]:
# digits to words
from num2words import num2words

def digits_to_ru_words(digits: str) -> str:
    n = int(digits)
    w = num2words(n, lang="ru")
    return normalize_ru_text(w)

tests = ["1000", "1005", "1010", "1100", "15000", "999999"]
for t in tests:
    print(t, "->", digits_to_ru_words(t))

1000 -> одна тысяча
1005 -> одна тысяча пять
1010 -> одна тысяча десять
1100 -> одна тысяча сто
15000 -> пятнадцать тысяч
999999 -> девятьсот девяносто девять тысяч девятьсот девяносто девять


In [12]:
UNITS = {
    "ноль": 0,
    "один": 1, "одна": 1, "одно": 1,
    "два": 2, "две": 2,
    "три": 3, "четыре": 4, "пять": 5, "шесть": 6, "семь": 7, "восемь": 8, "девять": 9,
}
TEENS = {
    "десять": 10, "одиннадцать": 11, "двенадцать": 12, "тринадцать": 13, "четырнадцать": 14,
    "пятнадцать": 15, "шестнадцать": 16, "семнадцать": 17, "восемнадцать": 18, "девятнадцать": 19,
}
TENS = {
    "двадцать": 20, "тридцать": 30, "сорок": 40, "пятьдесят": 50, "шестьдесят": 60,
    "семьдесят": 70, "восемьдесят": 80, "девяносто": 90,
}
HUNDREDS = {
    "сто": 100, "двести": 200, "триста": 300, "четыреста": 400,
    "пятьсот": 500, "шестьсот": 600, "семьсот": 700, "восемьсот": 800, "девятьсот": 900,
}
THOUSANDS = {"тысяча", "тысячи", "тысяч"}

In [13]:
def parse_0_999(tokens: List[str]) -> int:
    """
    Parse RU tokens for 0..999
    """
    total = 0
    i = 0
    while i < len(tokens):
        t = tokens[i]
        if t in HUNDREDS:
            total += HUNDREDS[t]
        elif t in TEENS:
            total += TEENS[t]
        elif t in TENS:
            total += TENS[t]
            # optional unit after tens
            if i + 1 < len(tokens) and tokens[i+1] in UNITS:
                total += UNITS[tokens[i+1]]
                i += 1
        elif t in UNITS:
            total += UNITS[t]
        # else: unknown token -> skip
        i += 1
    return total

In [14]:
def ru_words_to_int(text: str) -> int:
    """
    Convert normalized words to int
    """
    text = normalize_ru_text(text)
    if not text:
        raise ValueError("empty text")

    tokens = text.split()

    th_pos = None
    for idx, tok in enumerate(tokens):
        if tok in THOUSANDS:
            th_pos = idx
            break

    if th_pos is None:
        return parse_0_999(tokens)

    left = tokens[:th_pos]
    right = tokens[th_pos+1:]

    mult = parse_0_999(left) if left else 1
    if mult == 0:
        mult = 1

    value = mult * 1000 + parse_0_999(right)
    return value

In [15]:
examples = [
    "одна тысяча пять",   # 1005
    "тысяча пять",        # 1005
    "девять тысяч сто",   # 9100
    "двадцать одна тысяча сорок два",  # 21042
    "четыреста шестьдесят одна тысяча шестьсот", # 461600
    digits_to_ru_words("999999"),
]
for i in examples:
    print(i, "->", ru_words_to_int(i))

одна тысяча пять -> 1005
тысяча пять -> 1005
девять тысяч сто -> 9100
двадцать одна тысяча сорок два -> 21042
четыреста шестьдесят одна тысяча шестьсот -> 461600
девятьсот девяносто девять тысяч девятьсот девяносто девять -> 999999


#### preprocess train set text

In [16]:
# ru translate to dataset
train_df = train_df.copy()
dev_df = dev_df.copy()

train_df["ru_text"] = train_df["transcription"].map(digits_to_ru_words)
dev_df["ru_text"]   = dev_df["transcription"].map(digits_to_ru_words)

display(train_df[["filename","transcription","ru_text","spk_id"]].head(10))

,filename,transcription,ru_text,spk_id
0,train/0007c21c23.wav,139473,сто тридцать девять тысяч четыреста семьдесят три,spk_E
1,train/000bee1b1d.wav,992597,девятьсот девяносто две тысячи пятьсот девянос...,spk_B
2,train/001a718902.wav,500869,пятьсот тысяч восемьсот шестьдесят девять,spk_A
3,train/001e8e5565.wav,969908,девятьсот шестьдесят девять тысяч девятьсот во...,spk_C
4,train/001ee5be6b.wav,80484,восемьдесят тысяч четыреста восемьдесят четыре,spk_E
5,train/002006039c.wav,806551,восемьсот шесть тысяч пятьсот пятьдесят один,spk_E
6,train/0023aeb702.wav,834867,восемьсот тридцать четыре тысячи восемьсот шес...,spk_E
7,train/0025aa83a3.wav,433454,четыреста тридцать три тысячи четыреста пятьде...,spk_E
8,train/002ac85f40.wav,986140,девятьсот восемьдесят шесть тысяч сто сорок,spk_B
9,train/002d609255.wav,278484,двести семьдесят восемь тысяч четыреста восемь...,spk_C


In [17]:
def build_char_vocab(texts: List[str]) -> List[str]:
    """
    Build character vocabulary from training text only
    CTC tokens
    """
    chars = set()
    for t in texts:
        t = normalize_ru_text(t)
        chars.update(list(t))
    chars.add(" ")
    vocab = ["<blank>"] + sorted(chars)
    return vocab

In [91]:
VOCAB = build_char_vocab(train_df["ru_text"].tolist())
stoi = {c:i for i,c in enumerate(VOCAB)}
itos = {i:c for c,i in stoi.items()}

print("Vocab size:", len(VOCAB))
print("50 tokens:", VOCAB[:50])

Vocab size: 21
50 tokens: ['<blank>', ' ', 'а', 'в', 'д', 'е', 'и', 'к', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'ц', 'ч', 'ш', 'ы', 'ь', 'я']


In [92]:
def text_to_ids(text: str) -> List[int]:
    text = normalize_ru_text(text)
    return [stoi[c] for c in text if c in stoi]

def ids_to_text(ids: List[int]) -> str:
    s = "".join(itos[i] for i in ids if i in itos and itos[i] != "<blank>")
    return normalize_ru_text(s)

In [93]:
t = train_df["ru_text"].iloc[0]
ids = text_to_ids(t)
print("text:", t)
print("ids[:30]:", ids[:30])
print("roundtrip:", ids_to_text(ids))

text: сто тридцать девять тысяч четыреста семьдесят три
ids[:30]: [13, 14, 10, 1, 14, 12, 6, 4, 15, 2, 14, 19, 1, 4, 5, 3, 20, 14, 19, 1, 14, 18, 13, 20, 16, 1, 16, 5, 14, 18]
roundtrip: сто тридцать девять тысяч четыреста семьдесят три


#### torch loader

In [20]:
import torchaudio
import torchaudio.transforms as T

In [21]:
def load_audio_mono(path: Path) -> Tuple[torch.Tensor, int]:
    """
    Returns:
      wav: Tensor shape [1, num_samples] (mono)
      sr:  sample rate (int)
    """
    path = Path(path)
    assert path.exists(), f"Missing audio file: {path}"

    last_err = None
    for backend in ["ffmpeg", "soundfile", None]:
        try:
            if backend is None:
                wav, sr = torchaudio.load(str(path))
            else:
                wav, sr = torchaudio.load(str(path), backend=backend)
            # wav: [channels, time]
            if wav.dim() != 2:
                raise RuntimeError(f"Unexpected wav shape: {wav.shape}")
            if wav.size(0) > 1:
                wav = wav.mean(dim=0, keepdim=True)
            return wav, sr
        except Exception as e:
            last_err = e

    raise RuntimeError(f"Failed to load audio: {path}. Last error: {last_err}")

### augmentations

#### waveform-level augs

In [22]:
def rms(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    return torch.sqrt(torch.mean(x**2) + eps)

In [23]:
def random_gain(wav: torch.Tensor, min_gain_db: float = -6.0, max_gain_db: float = 6.0) -> torch.Tensor:  
    g_db = random.uniform(min_gain_db, max_gain_db)
    g = 10 ** (g_db / 20)
    return wav * g

In [24]:
def add_white_noise_snr(wav: torch.Tensor, snr_db: float) -> torch.Tensor:
    """
    Mix white noise to achieve target SNR (dB)
    wav: [1, T]
    """
    sig_rms = rms(wav)
    noise = torch.randn_like(wav)
    noise_rms = rms(noise)
    # SNR = 20*log10(sig_rms / (noise_rms * k))
    k = sig_rms / (10 ** (snr_db / 20) * noise_rms + 1e-8)
    return wav + noise * k

In [25]:
def time_dropout(wav: torch.Tensor, drop_prob: float = 0.5, max_drop_ms: int = 80, sr: int = 16000) -> torch.Tensor:
    """
    Rough packet-loss simulation: randomly zero a short segment
    """
    if random.random() > drop_prob:
        return wav
    T = wav.size(-1)
    max_drop = int(sr * (max_drop_ms / 1000.0))
    if max_drop < 1 or T < 10:
        return wav
    seg = random.randint(1, max_drop)
    start = random.randint(0, max(0, T - seg))
    out = wav.clone()
    out[..., start:start+seg] = 0.0
    return out

In [26]:
def random_time_shift(wav: torch.Tensor, shift_ms: int = 40, sr: int = 16000) -> torch.Tensor:
    """
    Circular shift by up to +/- shift_ms
    """
    max_s = int(sr * (shift_ms / 1000.0))
    if max_s < 1:
        return wav
    s = random.randint(-max_s, max_s)
    return torch.roll(wav, shifts=s, dims=-1)

In [27]:
def speed_perturb(wav: torch.Tensor, sr: int, p: float = 0.3, speeds=(0.9, 1.0, 1.1)) -> Tuple[torch.Tensor, int]:
    """
    Speed perturbation using SoX effects if available
    Keeps sample rate at `sr` by applying 'rate' after 'speed'
    """
    if random.random() > p:
        return wav, sr
    sp = random.choice(speeds)
    if sp == 1.0:
        return wav, sr
    try:
        effects = [
            ["speed", str(sp)],
            ["rate", str(sr)],
        ]
        wav2, sr2 = torchaudio.sox_effects.apply_effects_tensor(wav, sr, effects)
        # wav2: [1, T2]
        return wav2, sr2
    except Exception:
        return wav, sr

#### background noise mixing

In [28]:
@dataclass
class NoiseConfig:
    noise_dir: Optional[Path] = None   # noise audio dir
    p: float = 0.3                     # probability to mix noise
    snr_db_min: float = 5.0
    snr_db_max: float = 20.0

def list_audio_files(root: Path) -> List[Path]:
    exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a"}
    files = []
    for p in root.rglob("*"):
        if p.suffix.lower() in exts:
            files.append(p)
    return files

class NoiseMixer:
    def __init__(self, cfg: NoiseConfig):
        self.cfg = cfg
        self.noise_files = []
        if cfg.noise_dir is not None and Path(cfg.noise_dir).exists():
            self.noise_files = list_audio_files(Path(cfg.noise_dir))
        print(f"Noise files: {len(self.noise_files)}")

    def __call__(self, wav: torch.Tensor, sr: int) -> torch.Tensor:
        if not self.noise_files or random.random() > self.cfg.p:
            return wav

        nf = random.choice(self.noise_files)
        noise, nsr = load_audio_mono(nf)

        # resample noise to sr if needed
        if nsr != sr:
            noise = T.Resample(nsr, sr)(noise)

        # match lengths by random crop or pad
        Tsig = wav.size(-1)
        Tn = noise.size(-1)
        if Tn >= Tsig:
            start = random.randint(0, Tn - Tsig)
            noise = noise[..., start:start+Tsig]
        else:
            # loop-pad
            reps = math.ceil(Tsig / Tn)
            noise = noise.repeat(1, reps)[..., :Tsig]

        snr_db = random.uniform(self.cfg.snr_db_min, self.cfg.snr_db_max)

        sig_rms = rms(wav)
        n_rms = rms(noise)
        k = sig_rms / (10 ** (snr_db / 20) * n_rms + 1e-8)

        return wav + noise * k

### feature extractor

16 kHz log-mel + CMVN + SpecAugment

In [ ]:
@dataclass
class FeatureConfig:
    sample_rate: int = 16000
    n_mels: int = 80
    n_fft: int = 400         # 25 ms @ 16k
    win_length: int = 400
    hop_length: int = 160    # 10 ms @ 16k
    f_min: float = 20.0
    f_max: Optional[float] = 8000.0
    apply_cmvn: bool = True

@dataclass
class SpecAugmentConfig:
    enabled: bool = True
    freq_mask_param: int = 15
    time_mask_param: int = 35
    num_freq_masks: int = 2
    num_time_masks: int = 2

class LogMelFeatureExtractor(nn.Module):
    def __init__(self, feat_cfg: FeatureConfig, spec_cfg: SpecAugmentConfig):
        super().__init__()
        self.feat_cfg = feat_cfg
        self.spec_cfg = spec_cfg

        self.mel = T.MelSpectrogram(
            sample_rate=feat_cfg.sample_rate,
            n_fft=feat_cfg.n_fft,
            win_length=feat_cfg.win_length,
            hop_length=feat_cfg.hop_length,
            f_min=feat_cfg.f_min,
            f_max=feat_cfg.f_max,
            n_mels=feat_cfg.n_mels,
            power=2.0,
            center=True,
        )

        # SpecAugment masks (применяется к [B or C, F, T])
        self.freq_mask = T.FrequencyMasking(freq_mask_param=spec_cfg.freq_mask_param)
        self.time_mask = T.TimeMasking(time_mask_param=spec_cfg.time_mask_param)

    def forward(self, wav_16k: torch.Tensor, train: bool) -> torch.Tensor:
        """
        wav_16k: [1, T]
        returns feats: [T_frames, n_mels]
        """
        with torch.no_grad():
            mel = self.mel(wav_16k)                 # [1, n_mels, frames]
            feat = torch.log(mel + 1e-5)            # [1, n_mels, frames]

            if train and self.spec_cfg.enabled:
                x = feat
                for _ in range(self.spec_cfg.num_freq_masks):
                    x = self.freq_mask(x)
                for _ in range(self.spec_cfg.num_time_masks):
                    x = self.time_mask(x)
                feat = x

            # [1, n_mels, frames] -> [frames, n_mels]
            feat = feat.squeeze(0).transpose(0, 1).contiguous()

            if self.feat_cfg.apply_cmvn:
                mean = feat.mean(dim=0, keepdim=True)
                std = feat.std(dim=0, keepdim=True).clamp_min(1e-5)
                feat = (feat - mean) / std

            return feat.float()

### dataset loader

In [31]:
@dataclass
class AugmentConfig:
    enabled: bool = True
    p_white_noise: float = 0.2
    snr_db_min: float = 8.0
    snr_db_max: float = 25.0
    p_speed: float = 0.3
    p_time_dropout: float = 0.2
    p_shift: float = 0.2

class RussianNumbersASRDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        data_root: Path,
        feature_extractor: LogMelFeatureExtractor,
        augment_cfg: AugmentConfig,
        noise_mixer: Optional[NoiseMixer] = None,
        target_sample_rate: int = 16000,
        train: bool = True,
    ):
        self.df = df.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.fx = feature_extractor
        self.augment_cfg = augment_cfg
        self.noise_mixer = noise_mixer
        self.target_sr = target_sample_rate
        self.train = train

        self._resamplers: Dict[Tuple[int,int], T.Resample] = {}

    def _resample(self, wav: torch.Tensor, orig_sr: int) -> torch.Tensor:
        if orig_sr == self.target_sr:
            return wav
        key = (orig_sr, self.target_sr)
        if key not in self._resamplers:
            self._resamplers[key] = T.Resample(orig_freq=orig_sr, new_freq=self.target_sr)
        return self._resamplers[key](wav)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        rel = row["filename"]
        path = self.data_root / rel

        wav, sr = load_audio_mono(path)     # [1, T], sr
        wav = wav.float()

        # speed perturb (before resample)
        if self.train and self.augment_cfg.enabled:
            wav, sr = speed_perturb(wav, sr, p=self.augment_cfg.p_speed)

        wav = self._resample(wav, sr)
        sr = self.target_sr

        # waveform augs at 16k
        if self.train and self.augment_cfg.enabled:
            wav = random_gain(wav, -6, 6)
            if random.random() < self.augment_cfg.p_shift:
                wav = random_time_shift(wav, shift_ms=40, sr=self.target_sr)
            if random.random() < self.augment_cfg.p_time_dropout:
                wav = time_dropout(wav, drop_prob=1.0, max_drop_ms=80, sr=self.target_sr)

            if random.random() < self.augment_cfg.p_white_noise:
                snr = random.uniform(self.augment_cfg.snr_db_min, self.augment_cfg.snr_db_max)
                wav = add_white_noise_snr(wav, snr)

            if self.noise_mixer is not None:
                wav = self.noise_mixer(wav, self.target_sr)

        # features
        feats = self.fx(wav, train=self.train)  # [frames, n_mels]

        # targets (char ids)
        ru_text = row["ru_text"]
        y = torch.tensor(text_to_ids(ru_text), dtype=torch.long)

        return {
            "feats": feats,                     # [T, C]
            "feat_len": feats.size(0),
            "targets": y,                       # [U]
            "target_len": y.numel(),
            "ru_text": ru_text,
            "digits": row["transcription"],
            "spk_id": row.get("spk_id", None),
            "domain": row.get("domain", None),
            "path": str(rel),
        }

In [32]:
def ctc_collate_fn(batch: List[Dict]) -> Dict[str, torch.Tensor]:
    """
    Pads features to max T, concatenates targets (CTC style)
    """
    max_T = max(x["feat_len"] for x in batch)
    C = batch[0]["feats"].size(1)

    feats_pad = torch.zeros(len(batch), max_T, C, dtype=torch.float32)
    feat_lens = torch.tensor([x["feat_len"] for x in batch], dtype=torch.long)

    targets = torch.cat([x["targets"] for x in batch], dim=0)
    target_lens = torch.tensor([x["target_len"] for x in batch], dtype=torch.long)

    meta = {k: [x[k] for x in batch] for k in ["ru_text","digits","spk_id","domain","path"]}

    for i, x in enumerate(batch):
        T_i = x["feat_len"]
        feats_pad[i, :T_i] = x["feats"]

    return {
        "feats": feats_pad,          # [B, T, C]
        "feat_lens": feat_lens,      # [B]
        "targets": targets,          # [sumU]
        "target_lens": target_lens,  # [B]
        **meta
    }

In [33]:
feat_cfg = FeatureConfig(sample_rate=16000)
spec_cfg = SpecAugmentConfig(enabled=True)
augment_cfg = AugmentConfig(enabled=True)

fx = LogMelFeatureExtractor(feat_cfg, spec_cfg)

In [ ]:
NOISE_DIR = Path(os.environ.get("NOISE_DIR", "")).resolve()
noise_mixer = None
if str(NOISE_DIR) and NOISE_DIR.exists():
    noise_mixer = NoiseMixer(NoiseConfig(noise_dir=NOISE_DIR, p=0.3))

Noise files: 17400


In [35]:
train_ds = RussianNumbersASRDataset(
    train_df, DATA_ROOT, fx, augment_cfg, noise_mixer=noise_mixer, target_sample_rate=16000, train=True
)
dev_ds = RussianNumbersASRDataset(
    dev_df, DATA_ROOT, fx, augment_cfg, noise_mixer=None, target_sample_rate=16000, train=False
)

In [36]:
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True,
    collate_fn=ctc_collate_fn
)
dev_loader = torch.utils.data.DataLoader(
    dev_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True,
    collate_fn=ctc_collate_fn
)

In [37]:
batch = next(iter(train_loader))
{k: (v.shape if torch.is_tensor(v) else type(v)) for k,v in batch.items()}

/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(


{'feats': torch.Size([32, 394, 80]),
 'feat_lens': torch.Size([32]),
 'targets': torch.Size([1520]),
 'target_lens': torch.Size([32]),
 'ru_text': list,
 'digits': list,
 'spk_id': list,
 'domain': list,
 'path': list}

In [38]:
print("feats:", batch["feats"].shape)          # [B, T, C]
print("feat_lens:", batch["feat_lens"][:5].tolist())
print("targets total:", batch["targets"].shape)
print("target_lens:", batch["target_lens"][:5].tolist())

feats: torch.Size([32, 394, 80])
feat_lens: [366, 342, 263, 350, 301]
targets total: torch.Size([1520])
target_lens: [43, 55, 38, 55, 47]


In [39]:
i = 0
start = 0 if i == 0 else int(batch["target_lens"][:i].sum())
end = start + int(batch["target_lens"][i])
ids = batch["targets"][start:end].tolist()
print("ru_text (ref):", batch["ru_text"][i])
print("ru_text (from ids):", ids_to_text(ids))
print("digits (ref):", batch["digits"][i])

ru_text (ref): шестьсот тридцать пять тысяч шестьдесят три
ru_text (from ids): шестьсот тридцать пять тысяч шестьдесят три
digits (ref): 635063


### model

In [40]:
def conv_out_len(L: torch.Tensor, kernel: int, stride: int, pad: int, dilation: int = 1) -> torch.Tensor:
    return torch.floor((L + 2*pad - dilation*(kernel - 1) - 1) / stride + 1).to(torch.long)

In [ ]:
class SmallConformerCTC(nn.Module):
    """
    Lightweight Conformer-CTC:
      log-mel (80) -> 1D conv subsampling (x4) -> Conformer -> Linear -> CTC
    """
    def __init__(
        self,
        vocab_size: int,
        feat_dim: int = 80,
        d_model: int = 160,
        num_layers: int = 10,
        num_heads: int = 4,
        ffn_dim: int = 320,
        depthwise_conv_kernel_size: int = 31,
        dropout: float = 0.1,
    ):
        super().__init__()

        # 1D conv subsampling along time: stride=2 twice => ~4x shorter T
        k = 5
        p = k // 2
        self.subsample = nn.Sequential(
            nn.Conv1d(feat_dim, d_model, kernel_size=k, stride=2, padding=p),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(d_model, d_model, kernel_size=k, stride=2, padding=p),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.sub_k = k
        self.sub_p = p

        self.encoder = torchaudio.models.Conformer(
            input_dim=d_model,
            num_heads=num_heads,
            ffn_dim=ffn_dim,
            num_layers=num_layers,
            depthwise_conv_kernel_size=depthwise_conv_kernel_size,
            dropout=dropout,
            use_group_norm=False,
            convolution_first=False,
        )

        self.ctc_head = nn.Linear(d_model, vocab_size)

    def forward(self, feats: torch.Tensor, feat_lens: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        feats:     [B, T, 80]
        feat_lens: [B]
        returns:
          logits:    [B, T', vocab]
          out_lens:  [B]
        """
        x = feats.transpose(1, 2)  # [B, 80, T]
        x = self.subsample(x)      # [B, d_model, T']
        out_lens = feat_lens
        out_lens = conv_out_len(out_lens, kernel=self.sub_k, stride=2, pad=self.sub_p)
        out_lens = conv_out_len(out_lens, kernel=self.sub_k, stride=2, pad=self.sub_p)

        x = x.transpose(1, 2).contiguous()  # [B, T', d_model]
        x, out_lens = self.encoder(x, out_lens)  # conforms to torchaudio Conformer forward

        logits = self.ctc_head(x)  # [B, T', vocab]
        return logits, out_lens

#### param check

In [42]:
def count_params(model: nn.Module) -> tuple[int, int]:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [43]:
model = SmallConformerCTC(
    vocab_size=len(VOCAB),
    feat_dim=80,
    d_model=160,
    num_layers=10,
    num_heads=4,
    ffn_dim=320,
    depthwise_conv_kernel_size=31,
    dropout=0.1,
).to(DEVICE)

In [44]:
total, trainable = count_params(model)
print(f"params: {total/1e6:.3f}M | trainable: {trainable/1e6:.3f}M")

params: 4.127M | trainable: 4.127M


In [45]:
batch = next(iter(train_loader))

feats = batch["feats"].to(DEVICE)          # [B, T, 80]
feat_lens = batch["feat_lens"].to(DEVICE)  # [B]

with torch.no_grad():
    logits, out_lens = model(feats, feat_lens)

print("input feats:", feats.shape)
print("logits:", logits.shape)
print("in_lens[:5]:", feat_lens[:5].tolist())
print("out_lens[:5]:", out_lens[:5].tolist())

/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(


input feats: torch.Size([32, 409, 80])
logits: torch.Size([32, 103, 21])
in_lens[:5]: [297, 394, 304, 196, 325]
out_lens[:5]: [75, 99, 76, 49, 82]


### model train utils

#### greedy decode

In [46]:
@torch.no_grad()
def ctc_greedy_decode_batch(logits: torch.Tensor, out_lens: torch.Tensor, blank_id: int = 0) -> List[str]:
    """
    logits:   [B, T, V]
    out_lens: [B]
    returns list of decoded strings (RU text)
    """
    preds = logits.argmax(dim=-1).cpu().tolist()  # [B, T]
    out_lens = out_lens.cpu().tolist()

    texts = []
    for seq, L in zip(preds, out_lens):
        seq = seq[:L]
        collapsed = []
        prev = None
        for t in seq:
            if t != prev:
                collapsed.append(t)
            prev = t
        # remove blanks
        collapsed = [t for t in collapsed if t != blank_id]
        texts.append(ids_to_text(collapsed))
    return texts

In [47]:
def levenshtein(a: str, b: str) -> int:
    if a == b:
        return 0
    if len(a) == 0:
        return len(b)
    if len(b) == 0:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        cur = [i]
        for j, cb in enumerate(b, start=1):
            ins = cur[j-1] + 1
            dele = prev[j] + 1
            sub = prev[j-1] + (0 if ca == cb else 1)
            cur.append(min(ins, dele, sub))
        prev = cur
    return prev[-1]

In [48]:
def cer(ref: str, hyp: str) -> float:
    ref = str(ref)
    hyp = str(hyp)
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    return levenshtein(ref, hyp) / len(ref)

#### ru text to int pred

In [49]:
def ru_text_to_digits_str(pred_ru: str) -> str:
    """
    Convert model's RU text prediction -> digits string
    If parsing fails, return "" (gives CER=1 vs non-empty ref)
    """
    pred_ru = normalize_ru_text(pred_ru)
    if not pred_ru:
        return ""
    try:
        n = ru_words_to_int(pred_ru)
        n = int(max(0, min(999_999, n)))
        return str(n)
    except Exception:
        return ""

#### dev eval

In [50]:
@torch.no_grad()
def evaluate_ctc(model: nn.Module, loader, device: str = DEVICE) -> Dict:
    model.eval()

    rows = []
    for batch in tqdm(loader, desc="eval", leave=False):
        feats = batch["feats"].to(device)
        feat_lens = batch["feat_lens"].to(device)

        logits, out_lens = model(feats, feat_lens)
        pred_ru = ctc_greedy_decode_batch(logits, out_lens, blank_id=0)
        pred_digits = [ru_text_to_digits_str(t) for t in pred_ru]

        for i in range(len(pred_digits)):
            ref = batch["digits"][i]
            hyp = pred_digits[i]
            rows.append({
                "spk_id": batch["spk_id"][i],
                "domain": batch["domain"][i],
                "ref": ref,
                "hyp": hyp,
                "cer": cer(ref, hyp),
                "path": batch["path"][i],
                "pred_ru": pred_ru[i],
            })

    df = pd.DataFrame(rows)

    overall = float(df["cer"].mean())

    by_domain = df.groupby("domain")["cer"].mean().to_dict()
    cer_inD = float(by_domain.get("inD", np.nan))
    cer_ooD = float(by_domain.get("ooD", np.nan))

    if np.isfinite(cer_inD) and np.isfinite(cer_ooD) and (cer_inD + cer_ooD) > 0:
        hm = (2 * cer_inD * cer_ooD) / (cer_inD + cer_ooD)
    else:
        hm = np.nan

    by_spk = (df.groupby(["domain","spk_id"])["cer"]
                .mean()
                .reset_index()
                .sort_values("cer", ascending=False))

    return {
        "overall_cer": overall,
        "cer_inD": cer_inD,
        "cer_ooD": cer_ooD,
        "hm_cer": float(hm) if np.isfinite(hm) else np.nan,
        "by_spk": by_spk,
        "detail_df": df,
    }

### train loop

(AMP + gradient clipping + OneCycleLR)

In [51]:
from torch.amp import GradScaler, autocast

In [52]:
def make_optimizer_and_scheduler(model: nn.Module, train_loader, epochs: int):
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=2e-3,
        epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1,
        anneal_strategy="cos",
        div_factor=10.0,
        final_div_factor=50.0,
    )
    return opt, sch

In [53]:
def filter_invalid_ctc_batch(batch: Dict, out_lens: torch.Tensor) -> Optional[Dict]:
    """
    Remove samples where target_len > out_len (CTC cannot align)
    Returns filtered batch dict or None if nothing remains
    """
    tgt_lens = batch["target_lens"]
    valid = (tgt_lens.to(out_lens.device) <= out_lens).cpu().numpy().astype(bool)

    if valid.all():
        return batch
    keep_idx = np.where(valid)[0]
    if len(keep_idx) == 0:
        return None

    feats = batch["feats"][keep_idx]
    feat_lens = batch["feat_lens"][keep_idx]
    target_lens = batch["target_lens"][keep_idx]

    # rebuild concatenated targets
    targets_list = []
    start = 0
    for i in range(len(batch["target_lens"])):
        L = int(batch["target_lens"][i])
        if valid[i]:
            targets_list.append(batch["targets"][start:start+L])
        start += L
    targets = torch.cat(targets_list, dim=0)

    meta = {}
    for k in ["ru_text","digits","spk_id","domain","path"]:
        meta[k] = [batch[k][i] for i in keep_idx]

    return {
        "feats": feats,
        "feat_lens": feat_lens,
        "targets": targets,
        "target_lens": target_lens,
        **meta
    }

In [69]:
def train(model: nn.Module, train_loader, dev_loader, epochs: int = 30, device: str = DEVICE, ckpt_path: str = "best_ctc.pt"):
    ctc_loss_fn = nn.CTCLoss(blank=0, reduction="mean", zero_infinity=True)

    opt, sch = make_optimizer_and_scheduler(model, train_loader, epochs)
    scaler = GradScaler("cuda", enabled=(device == "cuda"))

    best_hm = float("inf")
    history = []
    first_epoch = True

    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        n_steps = 0

        for batch in tqdm(train_loader, desc=f"train e{epoch}", leave=False):
            feats = batch["feats"].to(device)
            feat_lens = batch["feat_lens"].to(device)
            targets = batch["targets"].to(device)
            target_lens = batch["target_lens"].to(device)

            opt.zero_grad(set_to_none=True)

            with autocast(device_type="cuda", enabled=(device == "cuda")):
                logits, out_lens = model(feats, feat_lens)

                # filter invalid alignments
                tmp = {
                    "feats": batch["feats"],
                    "feat_lens": batch["feat_lens"],
                    "targets": batch["targets"],
                    "target_lens": batch["target_lens"],
                    "ru_text": batch["ru_text"],
                    "digits": batch["digits"],
                    "spk_id": batch["spk_id"],
                    "domain": batch["domain"],
                    "path": batch["path"],
                }
                filtered = filter_invalid_ctc_batch(tmp, out_lens.detach())
                if filtered is None:
                    continue

                # re-run forward only if we filtered (to keep shapes consistent)
                if len(filtered["feat_lens"]) != len(batch["feat_lens"]):
                    feats = filtered["feats"].to(device)
                    feat_lens = filtered["feat_lens"].to(device)
                    targets = filtered["targets"].to(device)
                    target_lens = filtered["target_lens"].to(device)
                    logits, out_lens = model(feats, feat_lens)

                log_probs = logits.log_softmax(dim=-1)          # [B, T, V]
                log_probs = log_probs.transpose(0, 1)           # [T, B, V]

                loss = ctc_loss_fn(log_probs, targets, out_lens, target_lens)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            scaler.step(opt)
            scaler.update()
            sch.step()

            running += float(loss.item())
            n_steps += 1

        train_loss = running / max(1, n_steps)

        # evaluate
        dev_metrics = evaluate_ctc(model, dev_loader, device=device)

        hm = dev_metrics["hm_cer"]
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            **{k: dev_metrics[k] for k in ["overall_cer","cer_inD","cer_ooD","hm_cer"]},
        })

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | "
            f"dev CER overall={dev_metrics['overall_cer']:.4f} | "
            f"inD={dev_metrics['cer_inD']:.4f} ooD={dev_metrics['cer_ooD']:.4f} | "
            f"HM={dev_metrics['hm_cer']:.4f}"
        )
        new_row = pd.DataFrame([{
            'epoch': epoch,
            'train_loss': train_loss,
            'dev_cer_overall': dev_metrics['overall_cer'],
            'dev_cer_inD': dev_metrics['cer_inD'],
            'dev_cer_ooD': dev_metrics['cer_ooD'],
            'dev_hm_cer': dev_metrics['hm_cer']
        }])
        new_row.to_csv("train_metrics.csv", mode='a', header=first_epoch, index=False, float_format='%.4f')
        first_epoch = False
        
        # worst speakers (to diagnose overfitting to in-domain voices)
        # dev_metrics["by_spk"].to_csv("dev_metrics_by_spk.csv", index=True)
        # display(dev_metrics["by_spk"].head(10))

        if np.isfinite(hm) and hm < best_hm:
            best_hm = hm
            torch.save({
                "model_state": model.state_dict(),
                "vocab": VOCAB,
                "config": {
                    "feat_cfg": feat_cfg.__dict__,
                    "spec_cfg": spec_cfg.__dict__,
                },
                "metrics": dev_metrics,
                "history": history,
            }, ckpt_path)
            print(f"Saved best checkpoint to {ckpt_path} (best HM CER={best_hm:.4f})")

    return history

In [70]:
hist = train(model, train_loader, dev_loader, epochs=7, ckpt_path="best_ctc.pt")

train e1:   0%|          | 0/393 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
eval:   0%|          | 0/71 [00:00<?, ?it/s]               /home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: 

Epoch 01 | train_loss=0.1241 | dev CER overall=0.1158 | inD=0.0581 ooD=0.1366 | HM=0.0815
Saved best checkpoint to best_ctc.pt (best HM CER=0.0815)


train e2:   0%|          | 0/393 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
eval:   0%|          | 0/71 [00:00<?, ?it/s]               /home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: 

Epoch 02 | train_loss=0.1233 | dev CER overall=0.0959 | inD=0.0422 ooD=0.1153 | HM=0.0618
Saved best checkpoint to best_ctc.pt (best HM CER=0.0618)


train e3:   0%|          | 0/393 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
eval:   0%|          | 0/71 [00:00<?, ?it/s]               /home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: 

...

Epoch 05 | train_loss=0.0527 | dev CER overall=0.0475 | inD=0.0076 ooD=0.0619 | HM=0.0136
Saved best checkpoint to best_ctc.pt (best HM CER=0.0136)


train e6:   0%|          | 0/393 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
eval:   0%|          | 0/71 [00:00<?, ?it/s]               /home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: 

Epoch 06 | train_loss=0.0382 | dev CER overall=0.0391 | inD=0.0084 ooD=0.0501 | HM=0.0144


train e7:   0%|          | 0/393 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
eval:   0%|          | 0/71 [00:00<?, ?it/s]               /home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: 

Epoch 07 | train_loss=0.0331 | dev CER overall=0.0408 | inD=0.0081 ooD=0.0526 | HM=0.0141


#### load best check

In [72]:
CKPT_PATH = "best_ctc.pt"

ckpt = torch.load(
    CKPT_PATH, 
    map_location=DEVICE,
    weights_only=False
)

In [73]:
model = SmallConformerCTC(
    vocab_size=len(VOCAB),
    feat_dim=80,
    d_model=160,
    num_layers=10,
    num_heads=4,
    ffn_dim=320,
    depthwise_conv_kernel_size=31,
    dropout=0.1,
).to(DEVICE)

In [74]:
model.load_state_dict(ckpt["model_state"])
model.eval()

total, trainable = count_params(model)
print(f"Loaded checkpoint | Total params: {total/1e6:.3f}M")
print(f"Checkpoint dev metrics: {ckpt.get('metrics', {}).get('hm_cer', 'N/A')}")

Loaded checkpoint | Total params: 4.127M
Checkpoint dev metrics: 0.013555032197710828


In [90]:
dev_metrics_final = evaluate_ctc(model, dev_loader, device=DEVICE)

print(f"\n=== best check dev metrics ===")
print(f"  overall CER : {dev_metrics_final['overall_cer']:.4f}")
print(f"  inD CER     : {dev_metrics_final['cer_inD']:.4f}")
print(f"  ooD CER     : {dev_metrics_final['cer_ooD']:.4f}")
print(f"  harmonic mean CER      : {dev_metrics_final['hm_cer']:.4f}  (lower is better)")
display(dev_metrics_final["by_spk"])

eval:   0%|          | 0/71 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
                                                     


=== best check dev metrics ===
  overall CER : 0.0475
  inD CER     : 0.0076
  ooD CER     : 0.0619
  harmonic mean CER      : 0.0136  (lower is better)


,domain,spk_id,cer
7,ooD,spk_I,0.128706
9,ooD,spk_K,0.079232
8,ooD,spk_J,0.025678
6,ooD,spk_H,0.023855
5,inD,spk_F,0.023333
1,inD,spk_B,0.007000
3,inD,spk_D,0.006667
2,inD,spk_C,0.006667
0,inD,spk_A,0.002000
4,inD,spk_E,0.000000


### test eval

In [ ]:
class TestASRDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        data_root: Path,
        feature_extractor: LogMelFeatureExtractor,
        target_sample_rate: int = 16000,
    ):
        self.df = df.reset_index(drop=True)
        self.data_root = Path(data_root)
        self.fx = feature_extractor
        self.target_sr = target_sample_rate
        self._resamplers: Dict[Tuple[int, int], T.Resample] = {}

    def _resample(self, wav: torch.Tensor, orig_sr: int) -> torch.Tensor:
        if orig_sr == self.target_sr:
            return wav
        key = (orig_sr, self.target_sr)
        if key not in self._resamplers:
            self._resamplers[key] = T.Resample(orig_freq=orig_sr, new_freq=self.target_sr)
        return self._resamplers[key](wav)   # correctly applied

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        rel = str(row["filename"])
        path = self.data_root / rel

        wav, sr = load_audio_mono(path)
        wav = wav.float()
        wav = self._resample(wav, sr)

        feats = self.fx(wav, train=False)   # [T, 80], no SpecAugment

        return {
            "feats": feats,
            "feat_len": feats.size(0),
            "path": rel,
        }

In [76]:
def test_collate_fn(batch: List[Dict]) -> Dict:
    max_T = max(x["feat_len"] for x in batch)
    C = batch[0]["feats"].size(1)

    feats_pad = torch.zeros(len(batch), max_T, C, dtype=torch.float32)
    feat_lens = torch.tensor([x["feat_len"] for x in batch], dtype=torch.long)

    for i, x in enumerate(batch):
        T_i = x["feat_len"]
        feats_pad[i, :T_i] = x["feats"]

    paths = [x["path"] for x in batch]

    return {
        "feats": feats_pad,
        "feat_lens": feat_lens,
        "paths": paths,
    }

In [ ]:
test_csv = DATA_ROOT / "test.csv"
assert test_csv.exists(), f"Missing {test_csv}"

test_df = pd.read_csv(test_csv)

print("test_df columns:", test_df.columns.tolist())
print("test samples:", len(test_df))
display(test_df.head(5))

test_df columns: ['filename', 'ext', 'samplerate']
test samples: 2582


,filename,ext,samplerate
0,test/d2440788a9.mp3,mp3,16000
1,test/e247dbf761.mp3,mp3,16000
2,test/071f4a5be7.mp3,mp3,16000
3,test/798bd15db7.mp3,mp3,16000
4,test/58c0464ad5.mp3,mp3,16000


In [78]:
test_ds = TestASRDataset(
    test_df, DATA_ROOT, fx, target_sample_rate=16000
)

test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=32, shuffle=False, num_workers=2,
    pin_memory=True, collate_fn=test_collate_fn
)

In [79]:
tbatch = next(iter(test_loader))
print("test feats:", tbatch["feats"].shape)

/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(


test feats: torch.Size([32, 479, 80])


In [80]:
model.eval()

SmallConformerCTC(
  (subsample): Sequential(
    (0): Conv1d(80, 160, kernel_size=(5,), stride=(2,), padding=(2,))
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Conv1d(160, 160, kernel_size=(5,), stride=(2,), padding=(2,))
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
  )
  (encoder): Conformer(
    (conformer_layers): ModuleList(
      (0-9): 10 x ConformerLayer(
        (ffn1): _FeedForwardModule(
          (sequential): Sequential(
            (0): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
            (1): Linear(in_features=160, out_features=320, bias=True)
            (2): SiLU()
            (3): Dropout(p=0.1, inplace=False)
            (4): Linear(in_features=320, out_features=160, bias=True)
            (5): Dropout(p=0.1, inplace=False)
          )
        )
        (self_attn_layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(i

In [81]:
all_paths: List[str] = []
all_preds: List[str] = []      # integer string predictions
all_ru_preds: List[str] = []   # intermediate RU text

In [82]:
with torch.no_grad():
    for tbatch in tqdm(test_loader, desc="test inference"):
        feats = tbatch["feats"].to(DEVICE)
        feat_lens = tbatch["feat_lens"].to(DEVICE)

        logits, out_lens = model(feats, feat_lens)

        pred_ru = ctc_greedy_decode_batch(logits, out_lens, blank_id=0)
        pred_digits = [ru_text_to_digits_str(t) for t in pred_ru]

        all_paths.extend(tbatch["paths"])
        all_preds.extend(pred_digits)
        all_ru_preds.extend(pred_ru)

test inference:   0%|          | 0/81 [00:00<?, ?it/s]/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
/home/user/miniconda3/envs/sound_g1/lib/python3.11/site-packages/torchaudio/__init__.py:86: UserWarning: The 'backend' parameter is not used by TorchCodec AudioDecoder.
  return load_with_torchcodec(
test inference: 100%|██████████| 81/81 [00:12<00:00,  6.26it/s]


In [85]:
print("test predictions:", len(all_preds))
print("predictions sample:", all_paths[4], all_preds[4], all_ru_preds[4])

test predictions: 2582
predictions sample: test/58c0464ad5.mp3 861168 восемьсот шестьдесят одна тысяча сто шестьдесят восемь


#### postprocess

In [86]:
results_df = pd.DataFrame({
    "filename": all_paths,
    "transcription": all_preds,
    "pred_ru": all_ru_preds,
})
display(results_df.head(10))

,filename,transcription,pred_ru
0,test/d2440788a9.mp3,461694,четыреста шестьдесят одна тысяча шестьсот девя...
1,test/e247dbf761.mp3,207723,двести семь тысяч семьсот двадцать три
2,test/071f4a5be7.mp3,79187,семьдесят девять тысяч сто восемьдесят семь
3,test/798bd15db7.mp3,64048,шестьдесят четыре тысячи сорок восемь
4,test/58c0464ad5.mp3,861168,восемьсот шестьдесят одна тысяча сто шестьдеся...
5,test/2042774c60.mp3,499108,четыреста девяносто девять тысяч сто восемь
6,test/d96a2defb1.mp3,57929,дестьсот пятьдесят семь тысяч девятьсот двадца...
7,test/f86697a638.mp3,700833,семьсот сорасемь тысяч восемьсот тридцать три
8,test/d01bbbf7c1.mp3,762131,семьсот шестьдесят две тысячи сто тридцать один
9,test/525ee1542f.mp3,8829,восеятьсот восемтдесят восемь тысяч восемьсот ...


#### form csv

In [87]:
SUBMISSION_PATH = Path("submission.csv")

submission = results_df[["filename", "transcription"]].copy()
submission.to_csv(SUBMISSION_PATH, index=False)

In [88]:
print(f"Submission written to: {SUBMISSION_PATH}")
print(f"Shape: {submission.shape}")
display(submission.head(10))

Submission written to: submission.csv
Shape: (2582, 2)


,filename,transcription
0,test/d2440788a9.mp3,461694
1,test/e247dbf761.mp3,207723
2,test/071f4a5be7.mp3,79187
3,test/798bd15db7.mp3,64048
4,test/58c0464ad5.mp3,861168
5,test/2042774c60.mp3,499108
6,test/d96a2defb1.mp3,57929
7,test/f86697a638.mp3,700833
8,test/d01bbbf7c1.mp3,762131
9,test/525ee1542f.mp3,8829
